In [2]:
import pandas as pd
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.preprocessing import LabelEncoder

In [4]:
X_train = pd.read_csv('Datasets/IMDb_train.csv')
X_test = pd.read_csv('Datasets/IMDb_test.csv')
y_train = pd.read_csv('Datasets/y_train.csv')
y_test = pd.read_csv('Datasets/y_test.csv')

In [104]:
#####Cleaning and tokenizing textual data; IMDb description and title
stop_words = set(stopwords.words('english'))
contraction_mapping = {
    "wasn't": "was not",
    "didn't": "did not",
    "he's":"he is",
    "doesn't": "does not",
    "hadn't": "had not",
    "hasn't": "has not",
    "won't": "will not",
    "Don't": "Do not",
    "it's": "it is",
    "isn't": "is not",
    "can't": "cannot",
    "that's": "that is",
    "I'm": "I am",
    "I'll": "I will",
    "it'll": "it will",
    "couldn't": "could not",
    "they'll": "they will",
    "you'll": "you will",
    "wouldn't": "would not",
    "I've": "I have",
    "that's": "that is",
    "Ma'am": "madam",
    "you're": "you are",
    "she'll": "she will",
    "there's": "there is",
    "don't": "do not",
    "don't": "do not",
    "shan't": "shall not",
    "one's": "ones",
    "she's": "she is",
    "let's": "let us",
    "you'd": "you would",
    "Meanwhile, back o...":"",
    "&apos;":"'",
    "Capt.":"captain",
    "&quot;good vs.":"",
    "Valentine's Day":"Valentine's_Day",
    "&quot;":"'",
    " Ar...":"",
    "throug...":"through.",
    "Unite...":"United States.",
    " However, the Robot Dev...":"", 
    "tota...":"totally.",
    "dime...":"dimension.",
    "transporter cy...":"transporter.",
    "reme...":"remembers.",
    "begi...":"begins.",
}

def expand_contractions_in_series(series, contraction_mapping):
    """
    Expand contractions in a Pandas Series.
    """
    def expand_contractions(text):
        for word in word_tokenize(text):
            if word.lower() in contraction_mapping.items(): 
                text = text.replace(word, contraction_mapping[word])                    
        return text                 
    
    return series.apply(expand_contractions)

def clean_description(description):
    """
    Tokenizes, lowercases, removes non-alphanumeric tokens, and excludes stopwords.
    """
    if not isinstance(description, str):  
        description = ""
    return ' '.join(
        [word.lower() for word in word_tokenize(description) if word.isalnum() and word.lower() not in stop_words]
    )


X_train_expanded = X_train.copy()
X_test_expanded = X_test.copy()

# Remove contractions from descriptions
X_train_expanded = X_train.copy()  
X_test_expanded = X_test.copy()    

X_train_expanded['IMDb description'] = expand_contractions_in_series(X_train_expanded['IMDb description'], contraction_mapping)
X_test_expanded['IMDb description'] = expand_contractions_in_series(X_test_expanded['IMDb description'], contraction_mapping)
X_train_expanded['title'] = expand_contractions_in_series(X_train_expanded['title'], contraction_mapping)
X_test_expanded['title'] = expand_contractions_in_series(X_test_expanded['title'], contraction_mapping)

X_train_cleaned = X_train_expanded.copy()  
X_test_cleaned = X_test_expanded.copy()

X_train_cleaned['IMDb description'] = X_train_cleaned['IMDb description'].apply(clean_description)
X_test_cleaned['IMDb description'] = X_test_cleaned['IMDb description'].apply(clean_description)
X_train_cleaned['title'] = X_train_cleaned['title'].apply(clean_description)
X_test_cleaned['title'] = X_test_cleaned['title'].apply(clean_description)

print(X_train_expanded['IMDb description'])
print(X_train_expanded['title'])
print(X_train_cleaned['IMDb description'])
print(X_train_cleaned['title'])



0      Dax and Worf accompany Martok on his first com...
1      The Caretaker&apos;s remains resonate, which m...
2      Chief O&apos;Brien, who still doesn&apos;t par...
3      Capt. Picard finds himself shifting continuall...
4      The leader of a wagon train across the New Mex...
                             ...                        
399    Odo begins recovery from the deadly disease wh...
400    On a mission to an alien training mission, the...
401    Capt. Kirk obsessively hunts for a mysterious ...
402    When the crew finds a mysterious alien burial ...
403    The Prophets have told Sisko they do not appro...
Name: IMDb description, Length: 404, dtype: object
0      "Star Trek: Deep Space Nine" Soldiers of the E...
1      "Star Trek: Voyager" Cold Fire (TV Episode 199...
2      "Star Trek: Deep Space Nine" Armageddon Game (...
3      "Star Trek: The Next Generation" All Good Thin...
4      "The Twilight Zone" A Hundred Yards Over the R...
                             ...     

In [81]:
#####Labeling categorical data; genre, director, writer and stars
def label_encode_columns(df, categorical_columns):
    """
    Label encodes the categorical columns in the dataframe.
    """
    encoded_df = df.copy()  
    encoders = {}  

    for col in categorical_columns:
        encoder = LabelEncoder()
        encoded_df[col] = encoder.fit_transform(df[col].astype(str))  
        encoders[col] = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))

    return encoded_df, encoders

categorical_columns = ['genre', 'director', 'writer', 'stars']
encoded_X_train, Train_label_encoders = label_encode_columns(X_train_cleaned, categorical_columns)
encoded_X_test, Test_label_encoders = label_encode_columns(X_test_cleaned, categorical_columns)
print(Train_label_encoders)



{'genre': {"['Action', 'Adventure', 'Drama']": 0, "['Action', 'Adventure', 'Sci-Fi']": 1, "['Animation', 'Action', 'Adventure']": 2, "['Animation', 'Adventure', 'Comedy']": 3, "['Drama', 'Fantasy', 'Horror']": 4}, 'director': {"['Abner Biberman']": 0, "['Alan Crosland Jr.']": 1, "['Alexander Singer']": 2, "['Allan Eastman']": 3, "['Allan Kroeker']": 4, "['Allen H. Miner']": 5, "['Andrew Robinson']": 6, "['Anson Williams']": 7, "['Anton Leader']": 8, "['Avery Brooks']": 9, "['Bernard Girard']": 10, "['Bill Reed', 'Hal Sutherland']": 11, "['Boris Sagal']": 12, "['Bret Haaland', 'Rich Moore', 'Gregg Vanzo']": 13, "['Bret Haaland', 'Rich Moore']": 14, "['Bret Haaland']": 15, "['Brian Sheesley']": 16, "['Buzz Kulik']": 17, "['Chip Chalmers']": 18, "['Chris Loudon', 'Bret Haaland', 'Rich Moore']": 19, "['Chris Loudon']": 20, "['Chris Sauve', 'Gregg Vanzo', 'Bret Haaland']": 21, "['Cliff Bole']": 22, "['Corey Allen']": 23, "['Crystal Chesney']": 24, "['Dan Curry']": 25, "['David Alexander']":